# Tagalog trends: model comparison

Forecast each trend one step ahead and compare a naive baseline, the MLP from `google_trends.ipynb`,
ARIMA, XGBoost and a random forest. Every model sees the same previous `LOOKBACK` values and the same
chronological 60/20/20 train/validation/test split.

The MLP, XGBoost and random forest train on the GPU. ARIMA runs on the CPU (statsmodels has no GPU version).
Run section 8 of `google_trends.ipynb` first; it writes the series in `data/processed/`.

In [ ]:
# If packages are missing: %pip install "xgboost>=3.1,<4" "statsmodels>=0.14,<1"
import copy
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from IPython.display import display
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
# Stop here if there is no GPU: every learned model below must train on CUDA.
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select the Tagatrend GPU (.venv) kernel.")
DEVICE = torch.device("cuda:0")


# Read back which device a trained XGBoost model actually used ("cuda:0" or "cpu").
def xgb_device(model):
    return json.loads(model.get_booster().save_config())["learner"]["generic_param"]["device"]


# A tiny XGBoost fit proves the GPU build works before the real training starts.
check = xgb.XGBRegressor(n_estimators=5, device="cuda").fit(np.random.rand(100, 4), np.random.rand(100))
print("PyTorch:", torch.cuda.get_device_name(DEVICE), "| XGBoost trains on:", xgb_device(check))

In [ ]:
RESOLUTION = "daily"  # or "weekly" (Sana All and Pabebe Girls only)
LOOKBACK = 14 if RESOLUTION == "daily" else 12  # How many past values each prediction sees.
MIN_WINDOWS = 80  # Same minimum as google_trends.ipynb.
SEED = 42
TRENDS = {"sana_all": "Sana All", "pabebe_girls": "Pabebe Girls", "never_grow_old": "Never Grow Old"}
COLORS = {"Naive": "#52514e", "ARIMA": "#eb6834", "MLP": "#2a78d6", "XGBoost": "#1baf7a",
          "Random forest": "#eda100", "MLP (global)": "#2a78d6", "XGBoost (global)": "#1baf7a",
          "Random forest (global)": "#eda100"}


# Load one stitched series written by section 8 of google_trends.ipynb (None if it is missing).
def load_series(slug, resolution):
    path = Path("data/processed") / f"{slug}_{resolution}.csv"
    if not path.exists():
        return None
    return pd.read_csv(path, index_col=0, parse_dates=True)["interest"]


# One small panel per trend, to see what the models will be asked to forecast.
def plot_series(series_by_name):
    fig, axes = plt.subplots(len(series_by_name), 1, figsize=(10, 2.2 * len(series_by_name)), squeeze=False)
    for ax, (name, s) in zip(axes[:, 0], series_by_name.items()):
        ax.plot(s.index, s, color="#2a78d6", linewidth=1.5)
        ax.set(title=name, ylabel="Interest", ylim=(0, 105))
        ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()


series = {name: load_series(slug, RESOLUTION) for slug, name in TRENDS.items()}
series = {name: s for name, s in series.items() if s is not None}
if not series:
    raise FileNotFoundError("No data in data/processed/. Run section 8 of google_trends.ipynb first.")
plot_series(series)

In [ ]:
# Turn a series into supervised rows: the previous `lookback` values (x) -> the next value (y), scaled 0-1.
# Missing values (dropouts) are carried forward as inputs but never used as targets.
def make_windows(values, lookback):
    filled = pd.Series(values).ffill().fillna(0).to_numpy()
    targets = np.array([t for t in range(lookback, len(values)) if np.isfinite(values[t])])
    x = np.stack([filled[t - lookback:t] for t in targets]) / 100
    y = values[targets] / 100
    return x.astype(np.float32), y.astype(np.float32), targets


# Chronological 60% train / 20% validation / 20% test, so no model sees the future.
def split(n):
    a, b = int(n * 0.6), int(n * 0.8)
    return slice(0, a), slice(a, b), slice(b, n)

In [ ]:
# MLP: the same network as google_trends.ipynb (16 hidden units) trained on the GPU.
# Keeps the weights from the epoch with the lowest validation loss; returns a predict function.
def fit_mlp(x_train, y_train, x_val, y_val, epochs=300, patience=30):
    torch.manual_seed(SEED)
    x_train, y_train, x_val, y_val = (torch.as_tensor(a, device=DEVICE) for a in (x_train, y_train, x_val, y_val))
    model = torch.nn.Sequential(torch.nn.Linear(x_train.shape[1], 16), torch.nn.ReLU(),
                                torch.nn.Linear(16, 1)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
    best_loss, best_state, stale = float("inf"), None, 0
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = torch.nn.functional.mse_loss(model(x_train).squeeze(-1), y_train)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = torch.nn.functional.mse_loss(model(x_val).squeeze(-1), y_val).item()
        if val_loss < best_loss:  # New best: remember these weights.
            best_loss, best_state, stale = val_loss, copy.deepcopy(model.state_dict()), 0
        else:  # Stop after `patience` epochs without improvement.
            stale += 1
            if stale >= patience:
                break
    model.load_state_dict(best_state)

    def predict(x):
        with torch.no_grad():
            return model(torch.as_tensor(x, device=DEVICE)).squeeze(-1).cpu().numpy()
    return predict


# Wrap a trained XGBoost model as a predict function, after confirming it trained on the GPU.
def gpu_predictor(model):
    if not xgb_device(model).startswith("cuda"):
        raise RuntimeError("XGBoost did not train on the GPU.")
    rounds = getattr(model, "best_iteration", -1) + 1  # 0 means all trees (no early stopping)
    # Predicting through a DMatrix avoids XGBoost's CPU/GPU mismatch warning.
    return lambda x: model.get_booster().predict(xgb.DMatrix(x), iteration_range=(0, rounds))


# XGBoost: boosted trees on the GPU; stops adding trees when validation error stops improving.
def fit_xgboost(x_train, y_train, x_val, y_val):
    model = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.03, max_depth=4, subsample=0.8,
                             early_stopping_rounds=100, device="cuda", random_state=SEED)
    model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    return gpu_predictor(model)


# Random forest: XGBoost's random-forest mode, the same algorithm as scikit-learn's but on the GPU.
def fit_random_forest(x_train, y_train, x_val, y_val):
    model = xgb.XGBRFRegressor(n_estimators=500, max_depth=8, subsample=0.8, colsample_bynode=0.8,
                               device="cuda", random_state=SEED)
    model.fit(x_train, y_train)
    return gpu_predictor(model)


# ARIMA (CPU): choose (p, d, q) by AIC on the training part, then predict every next value
# from the observed past with those fixed parameters. Handles missing values natively.
def arima_one_step(values, train_end):
    y = values / 100
    best = None
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for order in [(p, d, q) for p in range(3) for d in range(2) for q in range(3)]:
            try:
                fit = ARIMA(y[:train_end], order=order).fit()
            except (ValueError, np.linalg.LinAlgError):
                continue
            if best is None or fit.aic < best.aic:
                best = fit
        return best.apply(y).predict(), best.model.order


FITTERS = {"MLP": fit_mlp, "XGBoost": fit_xgboost, "Random forest": fit_random_forest}

In [ ]:
# Fit every model on every trend and score its one-step test predictions.
# global_models=True also trains one MLP, XGBoost and random forest on all trends' training rows together.
def run_comparison(series_by_name, lookback, min_windows=MIN_WINDOWS, global_models=False):
    # 1. Windows and splits for each trend.
    prepared = {}
    for name, s in series_by_name.items():
        values = s.to_numpy(dtype=float)
        x, y, targets = make_windows(values, lookback)
        if len(y) < min_windows:
            print(f"{name}: only {len(y)} windows; skipped.")
            continue
        prepared[name] = (s, values, x, y, targets, split(len(y)))

    # 2. Global models: pool the training and validation rows of every trend.
    pooled = {}
    if global_models and len(prepared) > 1:
        parts = [(x, y, train, val) for _, _, x, y, _, (train, val, _) in prepared.values()]
        x_train = np.concatenate([x[train] for x, _, train, _ in parts])
        y_train = np.concatenate([y[train] for _, y, train, _ in parts])
        x_val = np.concatenate([x[val] for x, _, _, val in parts])
        y_val = np.concatenate([y[val] for _, y, _, val in parts])
        pooled = {f"{label} (global)": fit(x_train, y_train, x_val, y_val) for label, fit in FITTERS.items()}

    # 3. Local models per trend, then score everything on that trend's test rows.
    rows, test_predictions = [], {}
    for name, (s, values, x, y, targets, (train, val, test)) in prepared.items():
        arima, order = arima_one_step(values, targets[train][-1] + 1)
        print(f"{name}: ARIMA{order}, {len(y)} windows")
        predictions = {"Naive": x[test, -1], "ARIMA": arima[targets[test]]}
        for label, fit in FITTERS.items():
            predictions[label] = fit(x[train], y[train], x[val], y[val])(x[test])
        for label, predict in pooled.items():
            predictions[label] = predict(x[test])

        actual = y[test] * 100
        for model, predicted in predictions.items():
            predicted = np.clip(predicted, 0, 1) * 100  # Back to the 0-100 index.
            predictions[model] = predicted
            rows.append({"Trend": name, "Model": model, "MAE": np.abs(predicted - actual).mean(),
                         "RMSE": np.sqrt(((predicted - actual) ** 2).mean())})
        test_predictions[name] = (s.index[targets[test]], actual, predictions)

    results = pd.DataFrame(rows)
    # Skill: % improvement over the naive forecast (undefined when naive is already perfect).
    naive_mae = results.loc[results["Model"] == "Naive"].set_index("Trend")["MAE"].replace(0, np.nan)
    results["Skill vs naive (%)"] = 100 * (1 - results["MAE"] / results["Trend"].map(naive_mae))
    return results, test_predictions


# Print MAE and skill as Model x Trend tables.
def show_results(results):
    for column in ["MAE", "Skill vs naive (%)"]:
        table = results.pivot(index="Model", columns="Trend", values=column)
        print(column)
        display(table.reindex([m for m in COLORS if m in table.index]).round(2))


# One chart per trend: observed test values against every model's one-step predictions.
def plot_predictions(test_predictions, label):
    for name, (dates, actual, predictions) in test_predictions.items():
        fig, ax = plt.subplots(figsize=(10, 3.5))
        ax.plot(dates, actual, color="#0b0b0b", linewidth=1.5, label="Observed")
        for model, predicted in predictions.items():
            dashed = model == "Naive" or "global" in model
            ax.plot(dates, predicted, color=COLORS[model], linewidth=1.5,
                    linestyle="--" if dashed else "-", label=model)
        ax.set(title=f"{name}: one-step test predictions ({label})", ylabel="Interest (0-100)")
        ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
        ax.grid(axis="y", alpha=0.25)
        fig.autofmt_xdate()
        fig.tight_layout()
        plt.show()

## Initial results

In [ ]:
initial_results, initial_predictions = run_comparison(series, LOOKBACK)
show_results(initial_results)

In [ ]:
plot_predictions(initial_predictions, RESOLUTION)

## Improved run

The initial run has three data problems, not model problems:

- **Sana All:** about a third of each raw daily window is 0, while the other days are 30-60. Google reports
  a day as 0 when searches fall below its privacy threshold, so these zeros are "too few to report", not zero.
- **Pabebe Girls:** the daily window runs to April 2016, but the trend died in October 2015, so every test
  target is 0 and the naive forecast is trivially perfect.
- **Never Grow Old:** only ~85 windows. The decay falls entirely in validation, so early stopping halts
  the MLP and XGBoost almost untrained.

The four changes:

1. **Mask censored zeros:** a 0 becomes missing when every nonzero day in the previous 28 days is at least
   `CENSOR_FLOOR` points (past values only). This can also hide real zeros for up to 28 days after a trend dies.
2. **Trim Pabebe Girls** to June-October 2015, so its test period still has activity.
3. **Sana All weekly:** weekly rows have no censored zeros; run it as its own comparison.
4. **Global models:** one MLP, XGBoost and random forest trained on all daily trends together, so Never Grow
   Old can learn decay from the others. Because pooling supplies the data, the minimum drops to 60 windows.

In [ ]:
CENSOR_FLOOR = 5  # Index points; nonzero days in these series rarely fall below this.
IMPROVED_MIN_WINDOWS = 60


# Change 1: a 0 right after a run of clearly nonzero days is a below-threshold day, not zero interest.
# Looks only at the previous `days` values, so no future information is used.
def mask_censored_zeros(s, floor=CENSOR_FLOOR, days=28):
    smallest_recent = s.where(s > 0).shift(1).rolling(days, min_periods=1).min()
    censored = (s == 0) & (smallest_recent >= floor)
    return s.mask(censored), int(censored.sum())


improved = {}
for name, s in series.items():
    if name == "Pabebe Girls":
        s = s.loc["2015-06-01":"2015-10-31"]  # Change 2: keep the active months only.
    improved[name], masked = mask_censored_zeros(s)
    print(f"{name}: {masked} censored zeros masked, {int(improved[name].notna().sum())} rows left")

# Change 3: Sana All at weekly resolution.
sana_weekly = load_series("sana_all", "weekly")
plot_series({**improved, "Sana All (weekly)": sana_weekly})

In [ ]:
# Change 4: daily trends with global models, plus Sana All weekly on its own (12 weekly lags).
improved_results, improved_predictions = run_comparison(
    improved, LOOKBACK, min_windows=IMPROVED_MIN_WINDOWS, global_models=True)
weekly_results, weekly_predictions = run_comparison({"Sana All (weekly)": sana_weekly}, lookback=12)
all_improved = pd.concat([improved_results, weekly_results], ignore_index=True)
show_results(all_improved)

In [ ]:
plot_predictions(improved_predictions, "daily, improved")
plot_predictions(weekly_predictions, "weekly")

In [ ]:
# Best model per trend, before and after the changes.
def best_per_trend(results, run):
    best = results.loc[results["Model"] != "Naive"].sort_values("MAE").groupby("Trend").head(1)
    naive = results.loc[results["Model"] == "Naive"].set_index("Trend")["MAE"]
    return best.assign(Run=run, **{"Naive MAE": best["Trend"].map(naive)})[
        ["Run", "Trend", "Model", "MAE", "Naive MAE", "Skill vs naive (%)"]]


display(pd.concat([best_per_trend(initial_results, "initial"),
                   best_per_trend(all_improved, "improved")]).round(2))

**Reading the results.** Trends are persistent, so the naive forecast is hard to beat one step ahead;
positive skill means a model beat it. Google's sampling noise between pull days is about 6 points, so
smaller MAE differences are not meaningful. Initial and improved MAEs are not strictly comparable:
masking and trimming change which days are scored.